# Sustainability Data Analysis: Air Quality Index (AQI) in Indian Cities
### Assignment Submission by Vaibhav

This notebook contains a professional data cleaning, exploratory analysis, and visualization process on air quality datasets of major Indian cities from 2015 to 2020. The source file used is `city_day.csv`.

## 1. Import Libraries and Configuration
We import essential libraries like Pandas, Numpy, and Matplotlib. Visual styles are configured for professional layouts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plots style for premium and neat look
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'

## 2. Load Dataset & Inspect Basic Information
We load `city_day.csv` and output its shape, column names, data types, and initial null values.

In [ ]:
# Load raw dataset
df = pd.read_csv('city_day.csv')

# Display rows and columns
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

# Display columns and data types
print("\n--- Column Data Types ---")
print(df.dtypes)

# Display missing value count per column
print("\n--- Initial Missing Value Counts ---")
missing_counts = df.isnull().sum()
for col in df.columns:
    pct = (missing_counts[col] / len(df)) * 100
    print(f"{col:15}: {missing_counts[col]:5d} ({pct:.2f}%)")

## 3. Remove Duplicate Rows
We check if any duplicate entries exist. If found, we drop them to ensure dataset integrity.

In [ ]:
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates removed successfully.")
else:
    print("No duplicate rows present. Dataset is clean of duplicate records.")

## 4. Convert Date to Datetime Format
We convert the `Date` column from string format to a standard datetime format so we can perform temporal analyses.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
print(f"Converted Date Column Type: {df['Date'].dtype}")
print(f"Date Range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")

## 5. Handle Missing Values (Cleaning)
We implement the cleaning policy:
1. Drop any column containing more than **60% missing values**.
2. For remaining **numerical columns**, impute missing values with the column **median**.
3. For remaining **categorical columns**, impute missing values with the column **mode**.

In [ ]:
# Drop columns with > 60% missing values
missing_pcts = df.isnull().sum() / len(df)
cols_to_drop = missing_pcts[missing_pcts > 0.60].index.tolist()
print(f"Columns to drop (>60% missing): {cols_to_drop}")

df_clean = df.drop(columns=cols_to_drop)
print(f"Justification: Column {cols_to_drop} is dropped since {missing_pcts[cols_to_drop[0]]*100:.2f}% of its values are missing, which is too high to impute reliably.\n")

# Separate numeric and object types
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

# Impute numeric columns with median
print("--- Numerical Columns Imputation ---")
for col in num_cols:
    median_val = df_clean[col].median()
    nulls = df_clean[col].isnull().sum()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f" - {col:10} | Imputed {nulls:5d} nulls with median: {median_val:.2f}")

# Impute categorical columns with mode
print("\n--- Categorical Columns Imputation ---")
for col in cat_cols:
    mode_val = df_clean[col].mode()[0]
    nulls = df_clean[col].isnull().sum()
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f" - {col:10} | Imputed {nulls:5d} nulls with mode: '{mode_val}'")

# Final verification
print(f"\nTotal remaining missing values: {df_clean.isnull().sum().sum()}")

## 6. Save Cleaned Dataset to Excel
We save the cleaned dataset to `Vaibhav_CleanedDataset.xlsx`.

In [ ]:
excel_name = 'Vaibhav_CleanedDataset.xlsx'
df_clean.to_excel(excel_name, index=False)
print(f"Cleaned dataset saved successfully as: {excel_name}")

## 7. Generate Summary Statistics
We generate and inspect descriptive statistics for the numerical columns.

In [ ]:
print("--- Summary Statistics for Cleaned Air Quality Data ---")
df_clean[num_cols].describe()

## 8. Exploratory Analysis: Insights and Anomalies
We print the key patterns and anomalies discovered.

In [ ]:
print("=== TOP 3 INSIGHTS / PATTERNS ===")
print("1. Correlation analysis showing CO (r=0.68) and PM2.5 (r=0.66) correlate strongly with AQI.")
print("2. Over the years, the AQI has decreased from 212.5 in 2015 to 113.5 in 2020, with a dramatic drop in 2020 during the Covid-19 lockdown.")
print("3. Ahmedabad (452.1) and Delhi (259.5) are identified as the most polluted hotspots in terms of average AQI.")

print("\n=== 2 ANOMALIES IDENTIFIED ===")
print("1. Ahmedabad recorded a maximum AQI of 2049.0 on 2018-02-19, which is a major outlier compared to normal values.")
print("2. Ahmedabad also recorded a peak CO level of 175.81 mg/m3 on 2017-10-25, which represents an extreme outlier compared to the mean CO of 2.15 mg/m3.")

## 9. Visualizations
We generate the three requested plots and save them as PNGs.

In [ ]:
# Plot 1: AQI distribution histogram
plt.figure(figsize=(10, 5))
sns.histplot(df_clean['AQI'], bins=50, kde=True, color='#2c3e50', edgecolor='white', alpha=0.85)
plt.axvline(df_clean['AQI'].mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f"Mean AQI ({df_clean['AQI'].mean():.1f})")
plt.axvline(df_clean['AQI'].median(), color='#2ecc71', linestyle='-', linewidth=2, label=f"Median AQI ({df_clean['AQI'].median():.1f})")
plt.title('AQI Distribution across Indian Cities', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('AQI Value')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.savefig('aqi_distribution.png', dpi=300)
plt.show()

In [ ]:
# Plot 2: Top 10 cities by average AQI
plt.figure(figsize=(10, 5))
top_10 = df_clean.groupby('City')['AQI'].mean().sort_values(ascending=False).head(10)
sns.barplot(x=top_10.values, y=top_10.index, palette='viridis', hue=top_10.index, legend=False)
plt.title('Top 10 Indian Cities with Highest Average AQI (2015-2020)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Average AQI')
plt.ylabel('City')
for i, v in enumerate(top_10.values):
    plt.text(v + 5, i, f"{v:.1f}", va='center', fontweight='semibold')
plt.xlim(0, top_10.values[0] * 1.15)
plt.tight_layout()
plt.savefig('top_10_cities_aqi.png', dpi=300)
plt.show()

In [ ]:
# Plot 3: AQI trend over time
plt.figure(figsize=(12, 5))
df_clean['YearMonth'] = df_clean['Date'].dt.to_period('M')
monthly_trend = df_clean.groupby('YearMonth')['AQI'].mean()
monthly_trend.index = monthly_trend.index.to_timestamp()

plt.plot(monthly_trend.index, monthly_trend.values, color='#e67e22', linewidth=2.5, label='Monthly Average AQI')
plt.plot(monthly_trend.index, monthly_trend.rolling(6).mean(), color='#2980b9', linewidth=2, linestyle='--', label='6-Month Moving Average')
plt.title('Air Quality Index (AQI) Trend over Time (2015-2020)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Timeline')
plt.ylabel('Average AQI')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('aqi_trend_over_time.png', dpi=300)
plt.show()